In [6]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.stats import randint, uniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Create the plots directory (if it doesn't exist) for saving figures
os.makedirs("plots", exist_ok=True)

print("1) TRAIN / VALIDATION / TEST SPLIT")


data = load_breast_cancer()
X, y = data.data, data.target

# First split off the test set (touched only once, at the very end)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Then split remaining into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Class balance — train: {y_train.mean():.3f} | val: {y_val.mean():.3f} | test: {y_test.mean():.3f}")


print("2) CROSS-VALIDATION — KFold vs StratifiedKFold")


pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000))])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(pipe, X_train_full, y_train_full, cv=kf, scoring="roc_auc")
print(f"Plain KFold        — ROC-AUC: {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}")


print("3) STRATIFIED K-FOLD")


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(pipe, X_train_full, y_train_full, cv=skf, scoring="roc_auc")
print(f"StratifiedKFold     — ROC-AUC: {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}")

# Show class balance per fold to prove the point
print("\nClass balance per fold:")
for i, (_, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    print(f"  Fold {i+1}: positive rate = {y_train_full[val_idx].mean():.3f}")


print("4) GRID SEARCH")


param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5, scoring="roc_auc", n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("Best params (Grid):", grid_search.best_params_)
print("Best CV ROC-AUC   :", round(grid_search.best_score_, 4))
print("Total combinations tried:", len(grid_search.cv_results_["params"]))


print("5) RANDOMIZED SEARCH")


param_distributions = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 20),
    "max_features": uniform(0.3, 0.7)
}
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=25, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1
)
random_search.fit(X_train, y_train)
print("Best params (Random):", random_search.best_params_)
print("Best CV ROC-AUC     :", round(random_search.best_score_, 4))
print("Total combinations tried:", 25, "(vs. grid's exhaustive search)")


print("6) HYPERPARAMETER TUNING — evaluate the tuned model ONCE on the test set")


best_model = random_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f"Final test accuracy (touched only once): {test_score:.4f}")


print("7) DATA LEAKAGE — demonstrating the mistake and the fix")


# WRONG: fit scaler on all data (including test) before splitting
scaler_wrong = StandardScaler().fit(X)  # sees test data statistics!
X_scaled_wrong = scaler_wrong.transform(X)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_scaled_wrong, y, test_size=0.2,
                                                          random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=5000).fit(Xw_train, yw_train)
leaky_score = leaky_model.score(Xw_test, yw_test)

# RIGHT: fit scaler only on training data
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler_right = StandardScaler().fit(Xr_train)
Xr_train_s = scaler_right.transform(Xr_train)
Xr_test_s = scaler_right.transform(Xr_test)
clean_model = LogisticRegression(max_iter=5000).fit(Xr_train_s, yr_train)
clean_score = clean_model.score(Xr_test_s, yr_test)

print(f"Test accuracy WITH leakage (scaler fit on all data)   : {leaky_score:.4f}")
print(f"Test accuracy WITHOUT leakage (scaler fit on train only): {clean_score:.4f}")
print("(On this dataset the gap is small, but on smaller/noisier data leakage can")
print(" meaningfully inflate reported performance — the practice is what matters.)")


print("8) OVERFITTING vs UNDERFITTING — learning curves")


# Overfit-prone model: unregularized deep decision tree
overfit_model = DecisionTreeClassifier(max_depth=None, random_state=42)
# Underfit-prone model: a stump (max_depth=1)
underfit_model = DecisionTreeClassifier(max_depth=1, random_state=42)
# Reasonably fit model
good_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model, title in zip(
    axes,
    [underfit_model, overfit_model, good_model],
    ["Underfitting (depth=1 stump)", "Overfitting (unlimited depth tree)", "Good fit (tuned Random Forest)"]
):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train_full, y_train_full, cv=5, scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 8), random_state=42
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Train score")
    ax.plot(train_sizes, val_scores.mean(axis=1), "o-", label="Validation score")
    ax.set_title(title)
    ax.set_xlabel("Training set size")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.set_ylim(0.5, 1.02)

plt.tight_layout()
# Save to the "plots" folder (which we created)
plt.savefig("plots/23_learning_curves.png", bbox_inches="tight")
plt.close()
print("Saved learning curves plot to plots/23_learning_curves.png")

for model, name in [(underfit_model, "Underfit stump"), (overfit_model, "Overfit tree"), (good_model, "Tuned RF")]:
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    val_acc = model.score(X_val, y_val)
    print(f"{name:20s} — train acc: {train_acc:.4f} | val acc: {val_acc:.4f} | gap: {train_acc - val_acc:.4f}")


print("9) BIAS-VARIANCE TRADEOFF — polynomial regression demo")


# Simple 1D regression problem with noise, to visualize the classic tradeoff
np.random.seed(0)
x = np.sort(np.random.uniform(0, 1, 60))
y_true = np.sin(2 * np.pi * x)
y_noisy = y_true + np.random.normal(0, 0.2, x.shape)
X_1d = x.reshape(-1, 1)

x_tr, x_te, y_tr, y_te = train_test_split(X_1d, y_noisy, test_size=0.3, random_state=0)

degrees = [1, 4, 15]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_errors, test_errors = [], []

for ax, degree in zip(axes, degrees):
    poly_pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=degree)),
        ("linreg", LinearRegression())
    ])
    poly_pipe.fit(x_tr, y_tr)
    train_mse = np.mean((poly_pipe.predict(x_tr) - y_tr) ** 2)
    test_mse = np.mean((poly_pipe.predict(x_te) - y_te) ** 2)
    train_errors.append(train_mse)
    test_errors.append(test_mse)

    x_plot = np.linspace(0, 1, 200).reshape(-1, 1)
    ax.scatter(x_tr, y_tr, s=15, label="Train data")
    ax.plot(x_plot, poly_pipe.predict(x_plot), color="red", label=f"Degree {degree} fit")
    ax.plot(x_plot, np.sin(2 * np.pi * x_plot), color="green", linestyle="--", label="True function")
    ax.set_title(f"Degree {degree} | Train MSE={train_mse:.3f} Test MSE={test_mse:.3f}")
    ax.legend(fontsize=8)
    ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.savefig("plots/23_bias_variance.png", bbox_inches="tight")
plt.close()

print("Degree 1  (high bias / underfit)  — Train MSE: {:.4f} | Test MSE: {:.4f}".format(train_errors[0], test_errors[0]))
print("Degree 4  (good balance)          — Train MSE: {:.4f} | Test MSE: {:.4f}".format(train_errors[1], test_errors[1]))
print("Degree 15 (high variance/overfit) — Train MSE: {:.4f} | Test MSE: {:.4f}".format(train_errors[2], test_errors[2]))
print("\nSaved bias-variance plot to plots/23_bias_variance.png")

print("\nDone.")

1) TRAIN / VALIDATION / TEST SPLIT
Train: 341 | Val: 114 | Test: 114
Class balance — train: 0.628 | val: 0.623 | test: 0.632
2) CROSS-VALIDATION — KFold vs StratifiedKFold
Plain KFold        — ROC-AUC: 0.9945 +/- 0.0065
3) STRATIFIED K-FOLD
StratifiedKFold     — ROC-AUC: 0.9959 +/- 0.0050

Class balance per fold:
  Fold 1: positive rate = 0.626
  Fold 2: positive rate = 0.626
  Fold 3: positive rate = 0.626
  Fold 4: positive rate = 0.626
  Fold 5: positive rate = 0.626
4) GRID SEARCH
Best params (Grid): {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best CV ROC-AUC   : 0.9927
Total combinations tried: 12
5) RANDOMIZED SEARCH
Best params (Random): {'max_depth': 8, 'max_features': 0.30054513608871003, 'min_samples_split': 2, 'n_estimators': 559}
Best CV ROC-AUC     : 0.9909
Total combinations tried: 25 (vs. grid's exhaustive search)
6) HYPERPARAMETER TUNING — evaluate the tuned model ONCE on the test set
Final test accuracy (touched only once): 0.9474
7) DATA LEAKAGE — d